In [4]:
import pandas as pd
import re
from itertools import product

# ---------- Árbol de expresión ----------
class Node:
    """Nodo del árbol lógico."""
    def __init__(self, op=None, children=None, value=None):
        self.op = op          # "AND", "OR" o None
        self.children = children or []
        self.value = value    # código si es hoja

    def __repr__(self):
        if self.op:
            return f"({self.op}: {self.children})"
        return self.value


# ---------- Normalización + Tokenización ----------
def _normalize(expr: str) -> str:
    """
    Limpia la expresión:
    - Elimina ' Y' (error de origen).
    - Colapsa espacios.
    - Convierte A/O a AND/OR con límites de palabra.
    """
    # 1) eliminar ' Y' o standalone 'Y' (por si viene sin espacio final)
    expr = re.sub(r'(?<=\s)Y\b', '', expr)      # elimina " Y" preservando el espacio previo
    expr = re.sub(r'\bY\b', '', expr)           # por si viene como token aislado
    expr = re.sub(r'\s+', ' ', expr or "").strip()

    # 2) convertir operadores con límites de palabra (no afecta 'MAT1111', etc.)
    expr = re.sub(r'\bA\b', 'AND', expr)
    expr = re.sub(r'\bO\b', 'OR', expr)

    # 3) limpiar espacios extra otra vez
    expr = re.sub(r'\s+', ' ', expr).strip()
    return expr


def _tokenize(expr: str):
    """
    Convierte string en tokens: '(', ')', 'AND', 'OR', o códigos como 'MAT1111', 'INTE 00', 'CSV0040'.
    Permite códigos con espacio + dígitos (p.ej., 'INTE 00').
    """
    # Asegurar que la normalización esté aplicada
    expr = _normalize(expr)
    # Captura: paréntesis | AND | OR | códigos alfanuméricos con posible ' espacio + dígitos'
    tokens = re.findall(r'\(|\)|AND|OR|[A-Z0-9]+(?:\s[0-9]+)?', expr)
    return tokens


# ---------- Parser (precedencia: AND > OR) ----------
class _TokenStream:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def peek(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def consume(self, expected=None):
        tok = self.peek()
        if tok is None:
            return None
        if expected and tok != expected:
            raise ValueError(f"Se esperaba '{expected}' y llegó '{tok}'")
        self.pos += 1
        return tok


def _parse_expression(ts: _TokenStream) -> Node:
    """Expresión completa: OR de ANDs"""
    node = _parse_and(ts)
    while ts.peek() == 'OR':
        ts.consume('OR')
        right = _parse_and(ts)
        if node.op == 'OR':
            node.children.append(right)
        else:
            node = Node(op='OR', children=[node, right])
    return node


def _parse_and(ts: _TokenStream) -> Node:
    """AND de primarias"""
    node = _parse_primary(ts)
    while ts.peek() == 'AND':
        ts.consume('AND')
        right = _parse_primary(ts)
        if node.op == 'AND':
            node.children.append(right)
        else:
            node = Node(op='AND', children=[node, right])
    return node


def _parse_primary(ts: _TokenStream) -> Node:
    tok = ts.peek()
    if tok == '(':
        ts.consume('(')
        node = _parse_expression(ts)
        ts.consume(')')
        return node
    elif tok in (None, 'AND', 'OR', ')', '('):
        # Vacío o token inesperado: devolvemos hoja vacía para no romper
        return Node(value='')
    else:
        # Código (materia)
        ts.consume()
        return Node(value=tok)


# ---------- Expansión de combinaciones ----------
def _expand(node: Node):
    """
    Devuelve lista de combinaciones, cada combinación es lista de códigos.
    """
    if node.op is None:
        return [[node.value]] if node.value else [[]]

    if node.op == 'AND':
        parts = [_expand(c) for c in node.children]
        combos = product(*parts)
        return [sum(c, []) for c in combos]

    if node.op == 'OR':
        out = []
        for c in node.children:
            out.extend(_expand(c))
        return out

    return [[]]


# ---------- Función principal ----------
def procesar_prerrequisitos(df_reglas: pd.DataFrame, df_plan: pd.DataFrame) -> pd.DataFrame:
    """
    Procesa las reglas de prerrequisitos usando parser por combinaciones (árbol),
    genera columnas 'Opcion_Prereq_#' y valida contra 'MatCrso' del plan de estudio.

    Reglas:
    - 'O' = OR (opciones distintas → columnas distintas).
    - 'A' = AND (mismos requeridos → unidos con '&' en la misma celda).
    - Paréntesis respetados.
    - Se eliminan duplicados dentro de cada opción.
    - Se descartan opciones donde algún código no esté en df_plan['MatCrso'].
    - Se elimina el token erróneo 'Y' si aparece (no se reemplaza por AND).
    """
    # Set de códigos válidos
    asignaturas_validas = set(df_plan['MatCrso'].astype(str).str.strip().unique())

    resultados = []
    for _, row in df_reglas.iterrows():
        rule = row.get('prereq completo', None)
        if pd.isna(rule) or str(rule).strip() == '':
            resultados.append([])  # sin prerequisitos
            continue

        tokens = _tokenize(str(rule))
        ts = _TokenStream(tokens)
        tree = _parse_expression(ts)
        opciones = _expand(tree)

        # Limpiar cada opción: quitar vacíos, deduplicar preservando orden
        opciones_limpias = []
        for op in opciones:
            seq = [x.strip() for x in op if x and x.strip()]
            dedup = list(dict.fromkeys(seq))  # quita duplicados preservando orden
            if not dedup:
                continue
            # Validar contra plan (todas deben existir)
            if all(m in asignaturas_validas for m in dedup):
                opciones_limpias.append(" & ".join(dedup))

        # Deduplicar opciones iguales (p. ej., si la expresión genera repetidos)
        opciones_limpias = list(dict.fromkeys(opciones_limpias))
        resultados.append(opciones_limpias)

    # Construir columnas Opcion_Prereq_#
    max_len = max((len(r) for r in resultados), default=0)
    df_out = df_reglas.copy()
    for i in range(max_len):
        df_out[f'Opcion_Prereq_{i+1}'] = [r[i] if i < len(r) else "" for r in resultados]

    return df_out




In [5]:
def main():
    """
    Función principal para ejecutar el procesamiento
    """
    print("Procesador de Prerrequisitos Universitarios")
    print("=" * 50)
    
    # Opción para mostrar ejemplos o prueba compleja
        
    # Solicitar la ruta del archivo principal
    file_path = 'para trabajar pre req progv2.xlsx'#input("Ingresa la ruta del archivo principal de prerrequisitos (CSV o Excel): ").strip()
    
    if not file_path:
        print("No se proporcionó una ruta de archivo.")
        return None
    
    # Solicitar el archivo de asignaturas válidas
    courses_file_path ='asignaturas en plan de estudio v2.xlsx' #input("\nIngresa la ruta del archivo 'asignaturas en plan de estudio.xlsx' (o presiona Enter para omitir filtrado): ").strip()
    
    if not courses_file_path:
        print("⚠️ No se proporcionó archivo de asignaturas. Se procesarán todos los prerrequisitos sin filtrar.")
        courses_file_path = None
    
    
    # Procesar el archivo
    print(f"\nProcesando archivo: {file_path}")
    if courses_file_path:
        print(f"Archivo de asignaturas válidas: {courses_file_path}")
    
    df_req= pd.read_excel(file_path) if file_path and str(file_path).endswith(('.xlsx', '.xls')) else pd.read_csv(file_path)
    df_courses= pd.read_excel(courses_file_path) if courses_file_path and str(courses_file_path).endswith(('.xlsx', '.xls')) else pd.read_csv(courses_file_path) if courses_file_path else None

    #df_req= df_req[df_req["Smbarul_Key_Rule"]=="FIS1033"]
    df_result = procesar_prerrequisitos(df_req, df_courses)
    
    # Mostrar información sobre el resultado
    print(f"\n✅ Procesamiento completado!")
    print(f"Total de filas: {len(df_result)}")
    
    # Contar cuántas columnas de opciones se crearon
    option_cols = [col for col in df_result.columns if col.startswith('Opcion_Prereq_')]
    print(f"Columnas de opciones creadas: {len(option_cols)}")
    
    # Contar opciones válidas totales
    valid_options_count = 0
    for col in option_cols:
        valid_options_count += sum(1 for val in df_result[col] if val and str(val).strip())
    print(f"Total de opciones de prerrequisitos válidas: {valid_options_count}")
    
    # Mostrar filas con prerrequisitos (no vacías)
    non_empty_prereqs = df_result[df_result['prereq completo'].notna() & (df_result['prereq completo'] != '')]
    if len(non_empty_prereqs) > 0:
        print(f"\nFilas con prerrequisitos: {len(non_empty_prereqs)}")
        print("\nPrimeras 3 filas con prerrequisitos:")
        cols_to_show = []
        if 'Smbarul_Key_Rule' in df_result.columns:
            cols_to_show.append('Smbarul_Key_Rule')
        cols_to_show.extend(['prereq completo'] + option_cols[:5])  # Mostrar solo las primeras 5 columnas de opciones
        
        available_cols = [col for col in cols_to_show if col in df_result.columns]
        print(non_empty_prereqs[available_cols].head(3).to_string())
    
    # Guardar el resultado
    timestamp = pd.Timestamp.now().strftime("%Y-%m-%d%H%M%S")
    file_parts = file_path.rsplit('.', 1)
    if len(file_parts) == 2:
        output_path = file_parts[0] +timestamp+ '_procesado_v3.' + file_parts[1]
    else:
        output_path = file_path +timestamp+ '_procesado_v3.csv'
    
    if file_path.endswith('.csv') or not file_path.endswith(('.xlsx', '.xls')):
        df_result.to_csv(output_path, index=False)
    else:
        df_result.to_excel(output_path, index=False)
    
    print(f"\n💾 Archivo guardado como: {output_path}")
    
    return df_result
    
  

if __name__ == "__main__":
    main()

Procesador de Prerrequisitos Universitarios

Procesando archivo: para trabajar pre req progv2.xlsx
Archivo de asignaturas válidas: asignaturas en plan de estudio v2.xlsx

✅ Procesamiento completado!
Total de filas: 3101
Columnas de opciones creadas: 56
Total de opciones de prerrequisitos válidas: 2854

Filas con prerrequisitos: 2063

Primeras 3 filas con prerrequisitos:
    Smbarul_Key_Rule                                                              prereq completo Opcion_Prereq_1 Opcion_Prereq_2 Opcion_Prereq_3 Opcion_Prereq_4 Opcion_Prereq_5
341          IGL4992                          IGL4990 O HICR 001 O CICR 001 O ECCR 001 O  IGL0251         IGL4990         IGL0251                                                
342          IGL4965                                     IGL4962 O CIRI 001 O HIRI 001 O EXIR 001         IGL4962                                                                
343          IGL1350   HINI 001 O CINI 001 O EXIN 001 O ECCR 003 O HICR 003 O CICR 003 O  IGL